# W1-06: improved preparation and resume

Stop the old notebook cell once, keep the same CPU runtime, then run these cells in order. This version reuses the existing audio and imports completed v1 shard audits. Code is extracted into a separate temporary directory.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import io
import subprocess
import sys
import zipfile

drive.mount("/content/drive")

# Select the UPDATED adi20-preparation.zip downloaded from the answer.
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith(".zip")]
if len(archives) != 1:
    raise ValueError("Upload exactly one updated preparation ZIP")
install_root = Path("/content/adi20-upgrade")
with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
    archive.extractall(install_root)

code_root = install_root / "adi20-preparation"
data_root = Path("/content/drive/MyDrive/Arabic-Dialect-Identification-data")
if not (code_root / "src/data/stop_prepare.py").is_file():
    raise ValueError("This is the old ZIP. Upload the updated package containing stop_prepare.py")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(code_root / "requirements-data.txt"),
], check=True)
print("Updated code:", code_root)
print("Existing data:", data_root)


In [ ]:
# Stop matching old preparation workers and clear ONLY their empty stale lock.
revision = "c8da454c9f474bb075d3fe608940ee29e8ccfa43"
legacy_report = data_root / "reports" / "adi20_micro" / revision / "full-06cc7cf3bc843774"
stop_result = subprocess.run([
    sys.executable, str(code_root / "src/data/stop_prepare.py"),
    "--data-root", str(data_root),
    "--clear-lock", str(legacy_report / ".prepare.lock"),
], capture_output=True, text=True)
print(stop_result.stdout)
if stop_result.stderr:
    print(stop_result.stderr)
stop_result.check_returncode()
print("Ready to import the saved audits and resume.")


## Resume the full audit

This uses explicit **size verification** for the working Drive copy. It checks stored WAV existence and byte length, so it cannot detect same-size corruption. Source hashing, decoding, and audio audit checks remain enabled. To perform full stored-file verification later, change `size` to `sha256` in the command and rerun; completed valid clips are not decoded again. Source data, audio and audits stay on Drive; only one Parquet shard at a time is temporarily copied onto `/content`.


In [ ]:
# Fast working-copy mode: checks stored WAV sizes, not their full content hashes.
# Decoding and source hashing remain enabled. See the verification note above.
import signal
from datetime import datetime, timezone

command = [
    sys.executable, "-u", str(code_root / "src/data/prepare.py"),
    "--data-root", str(data_root), "--skip-download",
    "--resume-from", str(legacy_report),
    "--verification", "size",
    "--work-root", "/content/adi20-work",
]
log_dir = data_root / "reports"
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / ("prepare-v2-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + ".log")
print("Persistent log:", log_path, flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, cwd=code_root)
try:
    with log_path.open("a") as log:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
    exit_code = process.wait()
except KeyboardInterrupt:
    print("Stopping the preparation worker; completed shard audits remain saved.", flush=True)
    stopped = subprocess.run([
        sys.executable, str(code_root / "src/data/stop_prepare.py"),
        "--data-root", str(data_root),
    ], capture_output=True, text=True)
    print(stopped.stdout)
    if stopped.stderr:
        print(stopped.stderr)
    raise
print("Exit code:", exit_code)
if exit_code == 2:
    print("Completed with exclusions; inspect the printed audit report before training.")
elif exit_code != 0:
    raise RuntimeError("Preparation failed. The actual error is above and in the persistent log.")
